In [1]:

import pandas as pd
import requests
from pathlib import Path

# ==========================================
# 1. CONFIGURATION
# ==========================================

YEAR = 2026

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "dim_calendar.csv"
OUTPUT_PARQUET = OUTPUT_DIR / "dim_calendar.parquet"

# Generate complete ISO weeks covering YEAR
year_start = pd.Timestamp(f"{YEAR}-01-01")
year_end = pd.Timestamp(f"{YEAR}-12-31")

CALENDAR_START = (
    year_start -
    pd.Timedelta(days=year_start.dayofweek)
)

CALENDAR_END = (
    year_end +
    pd.Timedelta(days=6 - year_end.dayofweek)
)

print("Calendar start:", CALENDAR_START.date())
print("Calendar end:", CALENDAR_END.date())


# ==========================================
# 2. GENERATE COMPLETE CALENDAR
# ==========================================

dates = pd.date_range(
    start=CALENDAR_START,
    end=CALENDAR_END,
    freq="D"
)

calendar = pd.DataFrame({
    "attendance_date": dates
})

print("Calendar generated successfully!")
print("Total dates:", len(calendar))


# ==========================================
# 3. GET INDONESIAN HOLIDAYS
# ==========================================

print("\nFetching Indonesian holidays...")

# Fetch every year covered by the calendar
api_years = range(
    CALENDAR_START.year,
    CALENDAR_END.year + 1
)

holiday_frames = []

for api_year in api_years:

    api_url = (
        f"https://indonesian-holidays.ahrulsyamil.com/"
        f"holidays/{api_year}"
    )

    print(f"Fetching holidays: {api_year}")

    response = requests.get(
        api_url,
        timeout=30
    )

    response.raise_for_status()

    json_data = response.json()

    if "data" not in json_data:
        raise ValueError(
            f"Holiday API response does not contain 'data' "
            f"for {api_year}"
        )

    year_holidays = pd.DataFrame(
        json_data["data"]
    )

    if year_holidays.empty:
        print(f"WARNING: No holiday records for {api_year}")
        continue

    required_columns = [
        "date",
        "name",
        "type",
        "is_national"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in year_holidays.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing API columns for {api_year}: "
            f"{missing_columns}"
        )

    year_holidays = year_holidays[
        required_columns
    ].copy()

    holiday_frames.append(year_holidays)


if holiday_frames:

    holidays = pd.concat(
        holiday_frames,
        ignore_index=True
    )

else:

    holidays = pd.DataFrame(
        columns=[
            "date",
            "name",
            "type",
            "is_national"
        ]
    )

if not holidays.empty:

    holidays["date"] = pd.to_datetime(
        holidays["date"],
        errors="raise"
    ).dt.normalize()

    # Keep only dates within calendar range
    holidays = holidays[
        holidays["date"].between(
            CALENDAR_START,
            CALENDAR_END
        )
    ].copy()

    holidays["name"] = (
        holidays["name"]
        .fillna("")
        .astype("string")
        .str.strip()
    )

    holidays["type"] = (
        holidays["type"]
        .fillna("")
        .astype("string")
        .str.strip()
    )


# ==========================================
# 4. STANDARDIZE HOLIDAY FLAGS
# ==========================================

def convert_to_boolean(series):

    normalized = (
        series.astype("string")
        .str.strip()
        .str.lower()
    )

    true_values = [
        "true", "1", "yes", "y"
    ]

    false_values = [
        "false", "0", "no", "n", ""
    ]

    result = normalized.map(
        lambda x: (
            True if x in true_values
            else False if x in false_values
            else pd.NA
        )
    )

    return result.astype("boolean")


if not holidays.empty:

    holidays["is_national"] = convert_to_boolean(
        holidays["is_national"]
    )

    if holidays["is_national"].isna().any():
        raise ValueError(
            "Invalid values found in is_national"
        )


# ==========================================
# 5. GROUP HOLIDAYS BY DATE
# ==========================================

def combine_unique(values):

    unique_values = (
        values.astype("string")
        .dropna()
        .str.strip()
    )

    unique_values = unique_values[
        unique_values.ne("")
    ]

    return " | ".join(
        unique_values.drop_duplicates().tolist()
    )


if not holidays.empty:

    holidays_grouped = (
        holidays.groupby("date")
        .agg(
            holiday_name=(
                "name",
                combine_unique
            ),

            holiday_type=(
                "type",
                combine_unique
            ),

            is_national_holiday=(
                "is_national",
                "any"
            )
        )
        .reset_index()
        .rename(
            columns={
                "date": "attendance_date"
            }
        )
    )

else:

    holidays_grouped = pd.DataFrame(
        columns=[
            "attendance_date",
            "holiday_name",
            "holiday_type",
            "is_national_holiday"
        ]
    )


# ==========================================
# 6. MERGE CALENDAR + HOLIDAYS
# ==========================================

calendar = calendar.merge(
    holidays_grouped,
    on="attendance_date",
    how="left",
    validate="one_to_one"
)

calendar["holiday_name"] = (
    calendar["holiday_name"]
    .fillna("")
    .astype("string")
)

calendar["holiday_type"] = (
    calendar["holiday_type"]
    .fillna("")
    .astype("string")
)

calendar["is_national_holiday"] = (
    calendar["is_national_holiday"]
    .fillna(False)
    .astype("boolean")
)


# ==========================================
# 7. CREATE CALENDAR ATTRIBUTES
# ==========================================

day_mapping = {
    0: "Senin",
    1: "Selasa",
    2: "Rabu",
    3: "Kamis",
    4: "Jumat",
    5: "Sabtu",
    6: "Minggu"
}

month_mapping = {
    1: "Januari",
    2: "Februari",
    3: "Maret",
    4: "April",
    5: "Mei",
    6: "Juni",
    7: "Juli",
    8: "Agustus",
    9: "September",
    10: "Oktober",
    11: "November",
    12: "Desember"
}

calendar["year"] = (
    calendar["attendance_date"].dt.year
)

calendar["quarter"] = (
    calendar["attendance_date"].dt.quarter
)

calendar["month"] = (
    calendar["attendance_date"].dt.month
)

calendar["month_name"] = (
    calendar["month"].map(month_mapping)
)

calendar["day"] = (
    calendar["attendance_date"].dt.day
)

calendar["day_of_week"] = (
    calendar["attendance_date"].dt.dayofweek
)

calendar["day_name"] = (
    calendar["day_of_week"].map(day_mapping)
)


# ==========================================
# 8. CREATE WEEKEND ATTRIBUTES
# ==========================================

calendar["is_weekend"] = (
    calendar["day_of_week"] >= 5
)

calendar["day_type"] = "Weekday"

calendar.loc[
    calendar["is_weekend"],
    "day_type"
] = "Weekend"


# ==========================================
# 9. CREATE JOINT LEAVE ATTRIBUTE
# ==========================================

calendar["is_joint_leave"] = (
    calendar["holiday_type"]
    .str.contains(
        "joint_leave",
        case=False,
        na=False,
        regex=False
    )
)


# ==========================================
# 10. CREATE GENERAL HOLIDAY ATTRIBUTE
# ==========================================

calendar["is_any_holiday"] = (
    calendar["is_national_holiday"]
    |
    calendar["is_joint_leave"]
)


# ==========================================
# 11. CREATE WEEK ATTRIBUTES
# ==========================================

# Monday = beginning of the week
# Sunday = end of the week

calendar["week_start"] = (
    calendar["attendance_date"]
    -
    pd.to_timedelta(
        calendar["day_of_week"],
        unit="D"
    )
)

calendar["week_end"] = (
    calendar["week_start"]
    +
    pd.Timedelta(days=6)
)

# ISO week year
# Used to identify the year of the week,
# especially when the week crosses year boundaries.

iso_calendar = (
    calendar["attendance_date"]
    .dt.isocalendar()
)

calendar["week_year"] = (
    iso_calendar["year"].astype("int64")
)


# ==========================================
# 12. COUNT WEEKLY NATIONAL HOLIDAYS
# ==========================================

# Count only Monday-Friday national holidays.
# Weekend holidays do not reduce HO quota.

weekday_national_holidays = (
    calendar[
        calendar["is_national_holiday"]
        &
        calendar["day_of_week"].between(0, 4)
    ]
    .groupby("week_start")
    .size()
    .rename("weekday_national_holidays")
    .reset_index()
)

calendar = calendar.merge(
    weekday_national_holidays,
    on="week_start",
    how="left",
    validate="many_to_one"
)

calendar["weekday_national_holidays"] = (
    calendar["weekday_national_holidays"]
    .fillna(0)
    .astype("int64")
)


# ==========================================
# 13. COUNT WEEKLY JOINT LEAVES
# ==========================================

# Kept separate.
# Joint leave does not reduce quota yet.

weekday_joint_leaves = (
    calendar[
        calendar["is_joint_leave"]
        &
        calendar["day_of_week"].between(0, 4)
    ]
    .groupby("week_start")
    .size()
    .rename("weekday_joint_leaves")
    .reset_index()
)

calendar = calendar.merge(
    weekday_joint_leaves,
    on="week_start",
    how="left",
    validate="many_to_one"
)

calendar["weekday_joint_leaves"] = (
    calendar["weekday_joint_leaves"]
    .fillna(0)
    .astype("int64")
)


# ==========================================
# 14. CREATE WEEKLY ATTENDANCE QUOTA
# ==========================================

# HO employees:
# Normal weekly requirement = 5 days.
# National holidays on weekdays reduce the quota.

calendar["ho_weekly_attendance_quota"] = (
    5 - calendar["weekday_national_holidays"]
).clip(lower=0)


# Non-HO employees:
# Weekly attendance requirement = 6 days.
# Constant value for every calendar date.

calendar["non_ho_weekly_attendance_quota"] = 6


# ==========================================
# 15. DATA VALIDATION
# ==========================================

print("\n==========================================")
print("CALENDAR DATA VALIDATION")
print("==========================================")

expected_dates = len(
    pd.date_range(
        CALENDAR_START,
        CALENDAR_END,
        freq="D"
    )
)

if len(calendar) != expected_dates:
    raise ValueError(
        f"Unexpected date count: {len(calendar)}"
    )

duplicate_dates = (
    calendar["attendance_date"].duplicated().sum()
)

if duplicate_dates > 0:
    raise ValueError(
        f"Duplicate dates found: {duplicate_dates}"
    )

missing_dates = (
    calendar["attendance_date"].isna().sum()
)

if missing_dates > 0:
    raise ValueError(
        f"Missing dates found: {missing_dates}"
    )

if (
    calendar["attendance_date"].min()
    != CALENDAR_START
):
    raise ValueError("Incorrect calendar start date")

if (
    calendar["attendance_date"].max()
    != CALENDAR_END
):
    raise ValueError("Incorrect calendar end date")

# Verify Monday-Sunday week boundaries

if not calendar["week_start"].dt.dayofweek.eq(0).all():
    raise ValueError("Invalid week_start: must be Monday")

if not calendar["week_end"].dt.dayofweek.eq(6).all():
    raise ValueError("Invalid week_end: must be Sunday")

# Verify weekly quota consistency

if not (
    calendar.groupby("week_start")[
        "ho_weekly_attendance_quota"
    ].nunique().eq(1).all()
):
    raise ValueError("Inconsistent HO quota within a week")

if not calendar[
    "non_ho_weekly_attendance_quota"
].eq(6).all():
    raise ValueError("Invalid non-HO attendance quota")


print("Calendar start:", CALENDAR_START.date())
print("Calendar end:", CALENDAR_END.date())
print("Expected dates:", expected_dates)
print("Actual dates:", len(calendar))
print("Duplicate dates:", duplicate_dates)
print("Missing dates:", missing_dates)
print("Date range: VALID")
print("Weekly boundaries: VALID")
print("HO quota: VALID")
print("Non-HO quota: VALID")


# ==========================================
# 16. REORDER COLUMNS
# ==========================================

column_order = [
    "attendance_date",

    "year",
    "quarter",
    "month",
    "month_name",
    "day",

    "day_of_week",
    "day_name",
    "day_type",
    "is_weekend",

    "holiday_name",
    "holiday_type",
    "is_national_holiday",
    "is_joint_leave",
    "is_any_holiday",

    "week_start",
    "week_end",
    "week_year",

    "weekday_national_holidays",
    "weekday_joint_leaves",

    "ho_weekly_attendance_quota",
    "non_ho_weekly_attendance_quota"
]

calendar = calendar[column_order]


# ==========================================
# 17. EXPORT CALENDAR
# ==========================================

calendar.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

calendar.to_parquet(
    OUTPUT_PARQUET,
    index=False
)


# ==========================================
# 18. SUMMARY REPORT
# ==========================================

print("\n==========================================")
print("PROCESS 9 COMPLETED")
print("CALENDAR DIMENSION")
print("==========================================")


print("\nTOTAL DATES:")
print(len(calendar))


print("\nNATIONAL HOLIDAYS:")
print(
    calendar["is_national_holiday"].sum()
)


print("\nJOINT LEAVE:")
print(
    calendar["is_joint_leave"].sum()
)


print("\nWEEKENDS:")
print(
    calendar["is_weekend"].sum()
)


print("\nWEEKDAY NATIONAL HOLIDAYS:")
print(
    calendar.loc[
        calendar["is_national_holiday"]
        &
        calendar["day_of_week"].between(0, 4),
        "attendance_date"
    ].nunique()
)


print("\nWEEKDAY JOINT LEAVES:")
print(
    calendar.loc[
        calendar["is_joint_leave"]
        &
        calendar["day_of_week"].between(0, 4),
        "attendance_date"
    ].nunique()
)


print("\nWEEKLY ATTENDANCE QUOTA:")

weekly_summary = (
    calendar[
        [
            "week_start",
            "week_end",
            "week_year",
            "weekday_national_holidays",
            "weekday_joint_leaves",
            "ho_weekly_attendance_quota",
            "non_ho_weekly_attendance_quota"
        ]
    ]
    .drop_duplicates(subset=["week_start"])
    .sort_values("week_start")
)

print(
    weekly_summary.to_string(index=False)
)


print("\nOUTPUT FILES:")
print("CSV:", OUTPUT_CSV)
print("PARQUET:", OUTPUT_PARQUET)


print("\n==========================================")
print("PROCESS 9 SUCCESSFULLY FINISHED")
print("==========================================")


C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_11980\1903316788.py:23: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  pd.Timedelta(days=year_start.dayofweek)
C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_11980\1903316788.py:28: Pandas4Warning: Timestamp.dayofweek is deprecated and will be removed in a future version. Use Timestamp.day_of_week instead.
  pd.Timedelta(days=6 - year_end.dayofweek)


Calendar start: 2025-12-29
Calendar end: 2027-01-03
Calendar generated successfully!
Total dates: 371

Fetching Indonesian holidays...
Fetching holidays: 2025
Fetching holidays: 2026
Fetching holidays: 2027


C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_11980\1903316788.py:356: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  calendar["attendance_date"].dt.dayofweek
C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_11980\1903316788.py:581: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  if not calendar["week_start"].dt.dayofweek.eq(0).all():
C:\Users\PC IT - 1\AppData\Local\Temp\ipykernel_11980\1903316788.py:584: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  if not calendar["week_end"].dt.dayofweek.eq(6).all():



CALENDAR DATA VALIDATION
Calendar start: 2025-12-29
Calendar end: 2027-01-03
Expected dates: 371
Actual dates: 371
Duplicate dates: 0
Missing dates: 0
Date range: VALID
Weekly boundaries: VALID
HO quota: VALID
Non-HO quota: VALID

PROCESS 9 COMPLETED
CALENDAR DIMENSION

TOTAL DATES:
371

NATIONAL HOLIDAYS:
17

JOINT LEAVE:
8

WEEKENDS:
106

WEEKDAY NATIONAL HOLIDAYS:
13

WEEKDAY JOINT LEAVES:
8

WEEKLY ATTENDANCE QUOTA:
week_start   week_end  week_year  weekday_national_holidays  weekday_joint_leaves  ho_weekly_attendance_quota  non_ho_weekly_attendance_quota
2025-12-29 2026-01-04       2026                          1                     0                           4                               6
2026-01-05 2026-01-11       2026                          0                     0                           5                               6
2026-01-12 2026-01-18       2026                          1                     0                           4                               6
2026-01